In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_63_try_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 64 ###
def grab_subset_of_data_76(original_df, question_of_interest):
    # build a GPU‐side boolean mask for columns containing the question
    cols = original_df.columns.to_series()
    mask = cols.str.contains(question_of_interest)

    # select only those columns on the GPU
    subset_df = original_df.loc[:, mask]

    # generate simplified column names with GPU string ops
    new_names = (
        cols[mask]
        .str.split('-')
        .str.get(-1)
        .str.lstrip()
    )
    # assign the new names (small, metadata‐only CPU transfer)
    subset_df.columns = new_names.tolist()

    # drop rows where all tracked columns are null (GPU dropna)
    subset_df = subset_df.dropna(how='all')
    return subset_df

question_of_interest_cell76 = (
    'Do you use any tools to help monitor your machine learning models and/or experiments?'
)
tracking_df_2022 = grab_subset_of_data_76(
    responses_df_2022_cell10,
    question_of_interest_cell76
)
tracking_df_2022.info()